# 2. Hyperparameter tuning
Sweeps on the tuning DAGs (101–105), their results, and the candidate check that picks the config
for the main experiments in `3_main_experiments.ipynb`. Shared settings come from `1_mle.ipynb` (its `config` cell); this
notebook's own tuning and selection settings are tagged `config` too, so `3_main_experiments.ipynb` can load them.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import logging

import numpy as np
import pandas as pd
import torch.nn as nn

from ndbn.experiments import run_sweep, run_configs, save_chosen_configs
import ndbn.plotting as plotting
from ndbn.notebook_config import load_notebook_config

logging.getLogger("pgmpy").setLevel(logging.ERROR)

# shared settings (RESULTS_DIR, CARD, SAMPLE_SIZES, speed settings, ...) from 1_mle.ipynb
globals().update(load_notebook_config("1_mle.ipynb"))
print("results folder:", RESULTS_DIR)

results folder: results5-parallel


In [2]:
from IPython.display import HTML, display
display(HTML("""<style>
.cell-output-ipywidget-background { background-color: transparent !important; }
:root {
  --jp-widgets-color: var(--vscode-editor-foreground);
  --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>"""))

## Sweeps
Fixed graph size, varying only the estimator. Writes its own `sweep_tune.csv`, so it
cannot contaminate the main grid's `results.csv`. All tuning runs on separate DAGs
(101–105, `TUNE_DAG_OFFSET`); the main experiment's DAGs 1–5 are kept for final evaluation.

In [3]:
%pip install wandb   
# Offline alternative (sync later with `wandb sync`):  os.environ["WANDB_MODE"] = "offline"

Note: you may need to restart the kernel to use updated packages.


In [4]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Currently logged in as: kay-asp (kay-asp-wits-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
import torch.nn as nn

# All tuning (architecture sweep, no-ES diagnostic, hyperparameter sweep) runs on
# separate DAGs 101..100+SWEEP_DAGS from the same generator. The main experiment's
# DAGs (1..SIM) are kept for final evaluation only.
TUNE_DAG_OFFSET = 100
assert TUNE_DAG_OFFSET >= SIM, "tuning DAGs overlap the evaluation DAGs"

WANDB_PROJECT = "neural-cpds-tune"

SWEEP_CSV     = os.path.join(RESULTS_DIR, "sweep_tune.csv")
HIST_DIR      = os.path.join(RESULTS_DIR, "histories_tune")

SWEEP_N       = 20
SWEEP_ALPHA   = 0.5
SWEEP_DAGS    = 5
SWEEP_K       = [2, 6, 10]          # widen to [2, 4, 6, 8, 10] if time allows
SWEEP_SAMPLES = [600, 1400]

ARMS = [
    {"name": "nn_linear",     "hidden_dims": (),        "activation": None},
    {"name": "nn_32",         "hidden_dims": (32,),     "activation": nn.ReLU},
    {"name": "nn_64_32",      "hidden_dims": (64, 32),  "activation": nn.ReLU},
    {"name": "nn_128_64",     "hidden_dims": (128, 64), "activation": nn.ReLU},
    {"name": "nn_32_nowd",    "hidden_dims": (32,), "activation": nn.ReLU,
                              "weight_decay": 0.0},
    {"name": "nn_32_lr3e3",   "hidden_dims": (32,), "activation": nn.ReLU,
                              "lr": 3e-3},
    {"name": "nn_32_long",    "hidden_dims": (32,), "activation": nn.ReLU,
                              "n_epochs": 1000, "patience": 50},
]

In [6]:
sweep_df = run_sweep(
    arms=ARMS, k_values=SWEEP_K, n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD,
    val_frac=VAL_FRAC, n_epochs=N_EPOCHS,
    wandb_project=WANDB_PROJECT, sweep_csv=SWEEP_CSV, hist_dir=HIST_DIR,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES),   # main experiment's sample budget
    n_test=N_TEST,                                         # true KL on cached test.csv
    dag_offset=TUNE_DAG_OFFSET,                            # tuning DAGs, not evaluation
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

sweep runs:   0%|          | 0/210 [00:00<?, ?it/s]

KeyboardInterrupt: 

### No-early-stopping diagnostic
Patience > number of epochs, so nothing stops training. Shows how fast and how far each
architecture overfits at k = 10, N = 600 — compare `train_loss`, `val_loss` and `gap`
per epoch in W&B. Own project, CSV and history folder, so it can't mix with the main sweep.

In [ ]:
NO_ES_EPOCHS = 500
NO_ES_ARMS = [{**arm, "n_epochs": NO_ES_EPOCHS, "patience": 1000}
              for arm in ARMS if arm["name"] in ("nn_32", "nn_64_32", "nn_128_64")]

no_es_df = run_sweep(
    arms=NO_ES_ARMS, k_values=[10], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=[600], card=CARD,
    val_frac=VAL_FRAC, n_epochs=NO_ES_EPOCHS,
    wandb_project="neural-cpds-no-early-stopping-tune",
    sweep_csv=os.path.join(RESULTS_DIR, "sweep_no_es_tune.csv"),
    hist_dir=os.path.join(RESULTS_DIR, "histories_no_es_tune"),
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES),   # main experiment's sample budget
    n_test=N_TEST,                                         # true KL on cached test.csv
    dag_offset=TUNE_DAG_OFFSET,                            # tuning DAGs, not evaluation
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

### Hyperparameter sweep (W&B, Bayesian)
Tuning runs on **separate DAGs** (`dag_idx` > `TUNE_DAG_OFFSET`) drawn from the same generator,
never on the main experiment's DAGs. The chosen setting is then frozen and evaluated on the
main experiment's DAGs, so their test sets never influence any choice.

Each trial is one config scored on every tuning DAG × sample size for the given k. Because
the tuning DAGs are disjoint from evaluation, selecting on `mean_kl` (true KL on the tuning
DAGs' `test.csv`) is legitimate; `mean_val_loss` (validation-only) is the alternative. Both
are logged. AdamW, so `weight_decay` is decoupled and actually regularises. Start with
k = 10, then check how the frozen setting transfers to k = 2, 6.

In [7]:
from functools import partial
from ndbn.experiments import sweep_trial

HP_PROJECT      = "neural-cpds-hparams"
HP_TRIALS       = 50
HP_METRIC       = "mean_kl"      # or "mean_val_loss" (validation-only selection)
PARAMS          = {
        "hidden_dims":  {"values": ["128_64", "256_128", "128", "256", "32_32_32"
                                    "64_32_16", "64_64_64", "128_64_32",
                                    "128_128"]}, 
        "lr":           {"values": [1e-4, 3e-4, 6e-4, 1e-3]},     # on the lr snap grid (3-6)
        "dropout":      {"values": [0.0, 0.1, 0.3]},  # on the dropout snap grid (1-3)
        "weight_decay": {"value": 0.01},              # = FIXED
        "patience":     {"values": [20, 50]},
        "batch_size":   {"values": [32, 64, 128]},
        "optimizer":    {"value": "adamw"},
        "n_epochs":     {"value": 1000},
    }



In [ ]:
HP_K = 10
hp_sweep_id = 'p8lmbfxj' # wandb.sweep(HP_CONFIG, project=HP_PROJECT)   # or reuse an existing id to resume

HP_CONFIG = {
    "name": f"{HP_METRIC}_c{CARD}_k{HP_K}",
    "description": f"new architectures, card {CARD}, in-degree {HP_K}, tuning DAGs "
                   f"{TUNE_DAG_OFFSET + 1}-{TUNE_DAG_OFFSET + SWEEP_DAGS}, N = {SWEEP_SAMPLES}, "
                   f"snapped lr / dropout, weight decay fixed at the AdamW default",
    "method": "bayes",                # every combination, once
    "metric": {"name": HP_METRIC, "goal": "minimize"},
    "parameters": PARAMS,
}

run_trial = partial(
    sweep_trial, k_values=[HP_K], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    # one file for every sweep: each row records all its settings + sweep_id
    trial_csv=TRIALS_CSV,
    dag_offset=TUNE_DAG_OFFSET,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

from tqdm.auto import tqdm
bar = tqdm(total=HP_TRIALS, desc="sweep trials")

def trial():
    try:
        run_trial()
    finally:
        bar.update()          # counts even if a trial fails

wandb.agent(hp_sweep_id, project=HP_PROJECT, count=HP_TRIALS, function=trial)
bar.close()

sweep trials:   0%|          | 0/30 [00:00<?, ?it/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: ucbn6k8m with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.2
wandb: 	hidden_dims: 32
wandb: 	lr: 0.0006510754734969296
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.038074736311784166


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,2.15528
mean_val_loss,10.07331
sample_size,1400
std_kl,0.25164


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: odpkf784 with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 256_128
wandb: 	lr: 0.0008320415321964138
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 100
wandb: 	weight_decay: 0.007087669993708881


In [ ]:
HP_K = 6
hp_sweep_id = 'ey794fwc'  # or reuse an existing id to resume

HP_CONFIG = {
    "name": f"{HP_METRIC}_c{CARD}_k{HP_K}",
    "description": f"new architectures, card {CARD}, in-degree {HP_K}, tuning DAGs "
                   f"{TUNE_DAG_OFFSET + 1}-{TUNE_DAG_OFFSET + SWEEP_DAGS}, N = {SWEEP_SAMPLES}, "
                   f"snapped lr / dropout, weight decay fixed at the AdamW default",
    "method": "bayes",                # every combination, once
    "metric": {"name": HP_METRIC, "goal": "minimize"},
    "parameters": PARAMS,
}

run_trial = partial(
    sweep_trial, k_values=[HP_K], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    # one file for every sweep: each row records all its settings + sweep_id
    trial_csv=TRIALS_CSV,
    dag_offset=TUNE_DAG_OFFSET,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

from tqdm.auto import tqdm
bar = tqdm(total=HP_TRIALS, desc="sweep trials")

def trial():
    try:
        run_trial()
    finally:
        bar.update()          # counts even if a trial fails

wandb.agent(hp_sweep_id, project=HP_PROJECT, count=HP_TRIALS, function=trial)
bar.close()

Create sweep with ID: ey794fwc
Sweep URL: https://wandb.ai/kay-asp-wits-university/neural-cpds-hparams/sweeps/ey794fwc


sweep trials:   0%|          | 0/50 [00:00<?, ?it/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: lkee28ae with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 256_128
wandb: 	lr: 7.667716223070097e-05
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 5
wandb: 	weight_decay: 0.0001


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.9113
mean_val_loss,8.38638
sample_size,1400
std_kl,0.16738


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: lb30h9qy with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 64_32
wandb: 	lr: 0.0007043242037296014
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.0001


...

mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.59099
mean_val_loss,7.97251
sample_size,1400
std_kl,0.07795


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: 2bx6snxj with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 32
wandb: 	lr: 9.011172768233074e-05
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.0001


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0.465
mean_kl,1.07894
mean_val_loss,8.60762
sample_size,1400
std_kl,0.23531


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: b3t1w1t7 with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0
wandb: 	hidden_dims: 32
wandb: 	lr: 7.364868362695847e-05
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 10
wandb: 	weight_decay: 0.0001


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0.76
mean_kl,1.44201
mean_val_loss,8.99725
sample_size,1400
std_kl,0.31428


In [ ]:
HP_K = 2

HP_CONFIG = {
    "name": f"{HP_METRIC}_c{CARD}_k{HP_K}",
    "description": f"new architectures, card {CARD}, in-degree {HP_K}, tuning DAGs "
                   f"{TUNE_DAG_OFFSET + 1}-{TUNE_DAG_OFFSET + SWEEP_DAGS}, N = {SWEEP_SAMPLES}, "
                   f"snapped lr / dropout, weight decay fixed at the AdamW default",
    "method": "bayes",                # every combination, once
    "metric": {"name": HP_METRIC, "goal": "minimize"},
    "parameters": PARAMS,
}

hp_sweep_id = wandb.sweep(HP_CONFIG, project=HP_PROJECT)


run_trial = partial(
    sweep_trial, k_values=[HP_K], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    # one file for every sweep: each row records all its settings + sweep_id
    trial_csv=TRIALS_CSV,
    dag_offset=TUNE_DAG_OFFSET,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

from tqdm.auto import tqdm
bar = tqdm(total=HP_TRIALS, desc="sweep trials")

def trial():
    try:
        run_trial()
    finally:
        bar.update()          # counts even if a trial fails

wandb.agent(hp_sweep_id, project=HP_PROJECT, count=HP_TRIALS, function=trial)
bar.close()

Create sweep with ID: rmhp979u
Sweep URL: https://wandb.ai/kay-asp-wits-university/neural-cpds-hparams/sweeps/rmhp979u


sweep trials:   0%|          | 0/50 [00:00<?, ?it/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: fjt1sxyv with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0
wandb: 	hidden_dims: 256_128
wandb: 	lr: 0.0003
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.01


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.08416
mean_val_loss,7.46865
sample_size,1400
std_kl,0.02427


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: 33vepen5 with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 128_64_32128_128
wandb: 	lr: 0.0001
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 50
wandb: 	weight_decay: 0.01


## Hyperparameter sweep results
From `hparam_trials.csv` (every sweep; each row records all its settings and its `sweep_id`),
filtered with `TRIAL_FILTERS` (any recorded column). Weight decay was found to have a negligible effect
on KL, so it is fixed at the AdamW default of 0.01 (`FIXED`, set in the selection cell above); trials whose
settings are then identical (after rounding, or `SNAP`) are averaged automatically. One table per (in-degree, sample size): the configs
whose mean KL over the tuning DAGs is within one `kl_se` of that pair's best. Then one table of the
settings each pair's top configs share, and the configs that are in the top of more than one table, matched on `match_on` (add e.g. `"lr"`, rounded to 1
significant figure).

In [8]:
# Sweep-result selection (used by the sweep results and candidate cells below)
TRIAL_FILTERS = dict(card=CARD)   # any recorded column, e.g. alpha=SWEEP_ALPHA, sweep_id=...
N_SE          = 2                 # a config is "top" if its mean KL is within N_SE kl_se of the best

# Weight decay was found to have a negligible effect on KL (a 10x change in weight decay moves
# regret by <= ~0.2 percentage points with the other settings fixed), so it is fixed at the AdamW
# default 0.01: trials that then have identical settings are averaged automatically, and every
# chosen config runs with weight decay 0.01.
FIXED         = {"weight_decay": 0.01}
SNAP          = (("lr", (3,6)), ("dropout", (1,3)) )                # e.g. (("lr", (1, 3)),) to snap lr to 1e-4, 3e-4, 1e-3, ...

In [9]:
# hparam_trials.csv (all sweeps) plus any older per-k files
trials = plotting.load_trials(RESULTS_DIR)

# TRIAL_FILTERS, N_SE, FIXED and SNAP come from the selection cell above (weight decay fixed: negligible
# effect; trials that end up identical are averaged). Override here to explore, e.g. N_SE = 2.
tables = plotting.summarise_trials(trials, n_se=N_SE, fixed=FIXED, snap=SNAP, **TRIAL_FILTERS)

# per table: the settings all its top configs share (read down a column to compare tables)
shared  = plotting.shared_settings_table(tables)

# configs in the top of more than one table; match_on=(..., "lr") also matches lr (rounded)
common  = plotting.common_configs(tables, match_on=("hidden_dims", "dropout", "patience", "batch_size"))

# full settings of every top config
# lr rounded to 1 significant figure; weight decay fixed at the AdamW default for the runs
top_values = plotting.top_config_values(tables)   # uses the same FIXED / SNAP

in-degree 2, N = 600: best 0.1249 ± 0.0012 -> 1 of 3 configs (weight_decay fixed at 0.01) (lr snapped to 3-6 grid) (dropout snapped to 1-3 grid) within 2 kl_se


,run_name,mean_kl,kl_se,mean_val_loss,n_trials,hidden_dims,lr,dropout,patience,batch_size,stored_params,best_epoch_mean,fit_time_s,sweep_id
0,01_256x128_lr3e-4,0.1249,0.0012,7.497,2,"[256, 128]",0.0003,0.0,20,128,687400.0,58.035,23.8724,"rmhp979u, gqtytopa"


in-degree 2, N = 1400: best 0.0443 ± 0.0006 -> 1 of 3 configs (weight_decay fixed at 0.01) (lr snapped to 3-6 grid) (dropout snapped to 1-3 grid) within 2 kl_se


,run_name,mean_kl,kl_se,mean_val_loss,n_trials,hidden_dims,lr,dropout,patience,batch_size,stored_params,best_epoch_mean,fit_time_s,sweep_id
0,05_256x128_lr3e-4,0.0443,0.0006,7.4378,2,"[256, 128]",0.0003,0.0,20,128,687400.0,44.925,34.0855,"rmhp979u, gqtytopa"


in-degree 6, N = 600: best 0.7806 ± 0.0056 -> 6 of 55 configs (weight_decay fixed at 0.01) (lr snapped to 3-6 grid) (dropout snapped to 1-3 grid) within 2 kl_se


...

,in_degree,sample_size,n_top,hidden_dims,lr,dropout,patience,batch_size
0,2,600,1,"[256, 128]",≈3e-4,0.0,20,128
1,2,1400,1,"[256, 128]",≈3e-4,0.0,20,128
2,6,600,6,"[256, 128]",–,–,20,–
3,6,1400,2,"[128, 64]",≈3e-4,0.1,–,128
4,10,600,4,"[256, 128]",≈3e-4,0.0,–,–
5,10,1400,3,"[256, 128]",–,0.3,–,–


configs (matched on hidden_dims, dropout, patience, batch_size) in the top (within 2 kl_se) of >= 2 of 6 tables: 2


,hidden_dims,dropout,patience,batch_size,n_tables,top_in,n_trials
0,"[256, 128]",0.0,20,128,4,"k2 N600, k2 N1400, k6 N600, k10 N600",4
1,"[256, 128]",0.3,20,32,2,"k6 N600, k10 N1400",2


top configs (within 2 kl_se), lr snapped to 3-6 grid; dropout snapped to 1-3 grid; weight_decay fixed at 0.01 for runs, identical configs merged:


,run_name,in_tables,n_tables,n_trials,hidden_dims,activation,lr,dropout,patience,batch_size
0,01_256x128_lr3e-4,"k2 N600, k2 N1400, k6 N600, k10 N600",4,19,"[256, 128]",ReLU,0.00030,0.0,20,128
1,09_256x128_lr3e-4,k10 N600,1,7,"[256, 128]",ReLU,0.00030,0.0,20,32
2,35_256x128_lr2e-4,k6 N600,1,5,"[256, 128]",ReLU,0.00030,0.1,20,64
3,k6_256_128_25,k6 N600,1,5,"[256, 128]",ReLU,0.00030,0.1,20,128
4,10_256x128_lr3e-4,k10 N600,1,4,"[256, 128]",ReLU,0.00030,0.0,20,64
5,25_256x128_lr3e-4,k10 N600,1,3,"[256, 128]",ReLU,0.00030,0.0,100,32
6,40_256x128_lr3e-4,k10 N1400,1,3,"[256, 128]",ReLU,0.00030,0.3,50,32
7,49_256x128_lr6e-4,k10 N1400,1,3,"[256, 128]",ReLU,0.00060,0.3,20,64
8,k6_128_64_9,k6 N1400,1,2,"[128, 64]",ReLU,0.00030,0.1,50,128
9,k6_256_128_37,k6 N600,1,2,"[256, 128]",ReLU,0.00030,0.3,20,32


## Candidates
### Run chosen configs on all (in-degree, sample size)
Runs specific configs on the tuning DAGs (101–105) for the in-degrees in `CAND_SWEEPS` (k = 6, 10)
and every sample size in `SWEEP_SAMPLES`, so they are compared like for like (same DAGs, data and seeds). Evaluation DAGs stay untouched. `CANDIDATES` picks rows of the
top-config table above (`top_values.loc[[...]]`, using its row numbers) or is a list of config dicts.
They are fitted exactly like sweep
trials and **added to the tuning data** (`TRIALS_CSV`, `source = "candidates"`), so the sweep tables
above include them; in W&B, each config's k=6 run is added to the k=6 sweep and its k=10 run to the k=10
sweep, tagged `candidate`. Configs already
run are skipped on re-running. The comparison table below shows just these configs. Put the chosen config(s) into `CHOSEN_NN` below.

In [10]:
# in-degree -> existing W&B sweep: each config runs on these in-degrees, and its k=6 results go
# into the k=6 sweep, k=10 into the k=10 sweep (tagged "candidate")
CAND_SWEEPS = {6: "ey794fwc", 10: "p8lmbfxj"}

# Configs to run: rows of the top-config table above, picked by the row numbers it shows
# (or a list of config dicts written by hand). Names must be unique.
CANDIDATES = top_values.loc[[0, 1, 2, 3, 4, 13]]

# fitted exactly like sweep trials; rows are added to the tuning data (TRIALS_CSV, source="candidates",
# sweep_id of their in-degree's sweep). (config, in-degree) pairs already there are skipped on re-running.
cand = run_configs(
    CANDIDATES, k_values=list(CAND_SWEEPS), sweep_ids=CAND_SWEEPS, n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    trial_csv=TRIALS_CSV, dag_offset=TUNE_DAG_OFFSET,
    wandb_project="neural-cpds-hparams",      # same project as the sweeps; None to skip W&B
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

configs:   0%|          | 0/6 [00:00<?, ?it/s]

01_256x128_lr3e-4 k=[6]: already in hparam_trials.csv, skipped
01_256x128_lr3e-4 k=[10]: already in hparam_trials.csv, skipped
09_256x128_lr3e-4 k=[6]: already in hparam_trials.csv, skipped
09_256x128_lr3e-4 k=[10]: already in hparam_trials.csv, skipped
35_256x128_lr2e-4 k=[6]: already in hparam_trials.csv, skipped
35_256x128_lr2e-4 k=[10]: already in hparam_trials.csv, skipped
k6_256_128_25 k=[6]: already in hparam_trials.csv, skipped
k6_256_128_25 k=[10]: already in hparam_trials.csv, skipped
10_256x128_lr3e-4 k=[6]: already in hparam_trials.csv, skipped
10_256x128_lr3e-4 k=[10]: already in hparam_trials.csv, skipped


KeyboardInterrupt: 

### Candidate comparison
The configs run on every (in-degree, sample size) by the candidate cell above: mean KL over the
tuning DAGs. **Bold** = within `N_SE` standard errors of the best result *overall* for that
in-degree and sample size (all tuning trials, as in the sweep-results tables), not just the best candidate.

In [11]:
# bold = within N_SE SE of the best result overall (from the sweep-results tables above;
# without `tables`, the best of all tuning trials is used)
# index = row number in top_values (same settings), for choosing the final config below
cand_table = plotting.candidate_table(plotting.load_trials(RESULTS_DIR), n_se=4,
                                      tables=globals().get("tables"),
                                      top_values=globals().get("top_values"))
if cand_table.empty:
    print("no candidate runs yet: run the candidate cell above")

chosen configs on every (in-degree, sample size): mean KL over the tuning DAGs; bold = within 4 SE of the best result overall (all tuning trials) for that in-degree and sample size; index = row in top_values


,config,k6 N600,k6 N1400,k10 N600,k10 N1400,dropout,patience,batch_size,fit_time_s,params k6,params k10
top_idx,,,,,,,,,,,
0,07_256x128_lr3e-4 / 01_256x128_lr3e-4,0.7906,0.3730,2.2487,1.4395,0.0,20,128,44.5,719144,742696
3,k6_256_128_25,0.7815,0.3816,2.2755,1.4471,0.1,20,128,53.3,719144,742696
2,35_256x128_lr2e-4,0.7923,0.3750,2.2742,1.4546,0.1,20,64,63.0,719144,742696
4,10_256x128_lr3e-4,0.7824,0.3964,2.2455,1.4634,0.0,20,64,50.1,719144,742696
1,09_256x128_lr3e-4,0.7944,0.3887,2.2475,1.4524,0.0,20,32,70.4,719144,742696
8,k6_128_64_9,0.8067,0.3680,2.2946,1.4470,0.1,50,128,60.4,195752,207528


## Chosen configs
Pick the final NN config(s) for the main experiments by their `top_idx` in the candidate table above
(= row in `top_values`). The cell saves their settings to `chosen_nn.json` in the results folder;
`3_main_experiments.ipynb` reads that file and adds the `mle` and `nn_linear` baselines. The saved
choice stays fixed even if the indices change later (new trials, `N_SE` / `SNAP`), so re-run this
cell only when you want to change the final model.

In [12]:
# top_idx of the final config(s), from the candidate table above (= row in top_values)
CHOSEN_IDX = [0,8]

save_chosen_configs(top_values.loc[CHOSEN_IDX], CHOSEN_JSON)

saved 2 config(s) to results5-parallel\chosen_nn.json: ['01_256x128_lr3e-4', 'k6_128_64_9']


[{'name': '01_256x128_lr3e-4',
  'lr': 0.0003,
  'weight_decay': 0.01,
  'dropout': 0.0,
  'patience': 20,
  'batch_size': 128,
  'optimizer': 'adamw',
  'n_epochs': 1000,
  'hidden_dims': [256, 128],
  'activation': 'ReLU'},
 {'name': 'k6_128_64_9',
  'lr': 0.0003,
  'weight_decay': 0.01,
  'dropout': 0.1,
  'patience': 50,
  'batch_size': 128,
  'optimizer': 'adamw',
  'n_epochs': 1000,
  'hidden_dims': [128, 64],
  'activation': 'ReLU'}]